# Image Toolkit — AI Object Remover

This notebook runs **Stable Diffusion 1.5 Inpainting** as the AI service for the Image Toolkit FastAPI backend.

**Workflow:** upload an image and mask → run Stable Diffusion Inpainting → return the result through Gradio.

> Use test images that comply with Google Colab's policies. This notebook supports **Remove Object** only; the `reconstruct` mode has been removed.

## 1. Runtime setup

In Google Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU), then run the following cells.

In [1]:
!nvidia-smi

Tue Nov  3 07:10:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install dependencies

PyTorch is usually preinstalled in Google Colab, so this cell does not reinstall it. It installs the libraries required by the inpainting pipeline and Gradio.

In [2]:
%pip install -q -U diffusers transformers accelerate safetensors gradio gradio_client

## 3. Check GPU and dependencies

The model runs on a CUDA GPU. If no GPU is available, enable a GPU in the runtime settings before continuing.

In [3]:
import torch
import gradio as gr
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"Gradio: {gr.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU CUDA tidak tersedia. Aktifkan GPU melalui "
        "Runtime > Change runtime type > T4 GPU."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.11.0+cu130
Gradio: 6.31.0
CUDA available: True
GPU: Tesla T4


## 4. Load the inpainting model

The model remains `stable-diffusion-v1-5/stable-diffusion-inpainting`, as in the previous notebook.

In [4]:
from diffusers import AutoPipelineForInpainting

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"

pipe = AutoPipelineForInpainting.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

print(f"Model loaded: {MODEL_ID}")
print(f"Pipeline ready: {pipe is not None}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Model loaded: stable-diffusion-v1-5/stable-diffusion-inpainting
Pipeline ready: True


## 5. Inpainting function

Implementation notes:
- The input image is processed in RGB format and the mask in grayscale.
- The mask is binarized: white (`255`) marks the area that can be modified, while black (`0`) preserves the area.
- Input dimensions are adjusted to multiples of 8 for model compatibility.
- AI-generated content is applied only inside the mask. Pixels outside the mask are taken from the original image.
- No user prompt is used; the model uses a fixed base prompt.

In [5]:
from PIL import Image

INFERENCE_STEPS = 30
GUIDANCE_SCALE = 7.0
SEED = 42

BASE_PROMPT = (
    "realistic detailed image, natural texture, "
    "consistent lighting, seamless background"
)

NEGATIVE_PROMPT = (
    "blurry, distorted, stretched, low quality, artifacts"
)


def run_inpainting(image, mask, mode):
    """Remove the selected region using Stable Diffusion inpainting."""

    if image is None or mask is None:
        raise gr.Error("Both the image and mask are required.")

    # Only the remove-object mode is supported.
    if mode != "remove":
        raise gr.Error("Invalid processing mode. Only 'remove' is supported.")

    image = image.convert("RGB")
    original_size = image.size

    # Stable Diffusion dimensions should be multiples of 8.
    width = max(8, round(image.width / 8) * 8)
    height = max(8, round(image.height / 8) * 8)
    processing_size = (width, height)

    image_input = image.resize(
        processing_size,
        Image.Resampling.LANCZOS,
    )

    mask_input = mask.convert("L").resize(
        processing_size,
        Image.Resampling.NEAREST,
    )

    # White = editable region; black = preserved region.
    mask_input = mask_input.point(
        lambda pixel: 255 if pixel > 127 else 0
    )

    generator = torch.Generator(device="cuda").manual_seed(SEED)

    try:
        with torch.inference_mode():
            generated = pipe(
                prompt=BASE_PROMPT,
                negative_prompt=NEGATIVE_PROMPT,
                image=image_input,
                mask_image=mask_input,
                num_inference_steps=INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                strength=1.0,
                generator=generator,
            ).images[0]
    except Exception as exc:
        raise gr.Error(f"Inpainting failed: {exc}") from exc

    generated = generated.convert("RGB").resize(
        processing_size,
        Image.Resampling.LANCZOS,
    )

    # Composite generated pixels only inside the selected mask.
    result = Image.composite(
        generated,
        image_input,
        mask_input,
    )

    return result.resize(
        original_size,
        Image.Resampling.LANCZOS,
    )


## 6. Launch the Gradio API

The API input order is:
1. `image`
2. `mask`
3. `mode`

The API name remains **`/run_inpainting`**. The FastAPI backend can continue calling the same endpoint after its request is updated to send these three inputs. `share=True` creates a temporary public URL for connecting the local backend to the Colab runtime.

In [6]:
demo = gr.Interface(
    fn=run_inpainting,
    inputs=[
        gr.Image(type="pil", label="Original image"),
        gr.Image(type="pil", image_mode="L", label="Black-and-white mask"),
        gr.Radio(
            choices=[("Remove object", "remove")],
            value="remove",
            label="Processing mode",
        ),
    ],
    outputs=gr.Image(type="pil", label="Inpainting result"),
    title="Image Toolkit — AI Object Remover",
    description="Remove a selected region using Stable Diffusion inpainting.",
    api_name="run_inpainting",
)

demo.launch(
    share=True,
    debug=False,
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6f9ebc20fa6d6912a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Connect to the FastAPI backend

After Gradio starts, Colab will display a temporary public URL in the format `https://....gradio.live`.

Copy that URL into `backend/.env`:

```env
GRADIO_URL=https://your-gradio-session.gradio.live
```

Then restart the FastAPI backend so it reloads the environment variable.

**Important:**
- The Gradio URL changes when the Colab session/runtime is restarted.
- Do not commit the `.env` file or share an active session URL publicly.
- Colab runtimes can disconnect or stop; this notebook is not a 24/7 API host.
- Stop the runtime when you are finished using it.